# NeuroState Qwen Causal Intervention v0.9

Inspects which tokenizer tokens correspond to the v0.8 token-position map. This notebook does not load the language model; it only loads the tokenizer.

In [ ]:
!pip -q install 'transformers>=4.40'


In [ ]:
from pathlib import Path
WORK=Path('/content/neurostate_qwen_causal_intervention_v0_9')
WORK.mkdir(parents=True,exist_ok=True)


In [ ]:
(WORK/'intervention_core.py').write_text('from __future__ import annotations\n\nimport math\nimport random\nfrom typing import Sequence\n\nPROMPTS = (\n    "Write one short sentence about opening a window.",\n    "Write one short sentence about preparing tea.",\n    "Write one short sentence about an empty notebook.",\n    "Write one short sentence about walking to a station.",\n    "Write one short sentence about rain on a roof.",\n    "Write one short sentence about arranging a desk.",\n    "Write one short sentence about a quiet library.",\n    "Write one short sentence about starting a small task.",\n    "Write one short sentence about watching clouds.",\n    "Write one short sentence about tying a shoelace.",\n    "Write one short sentence about a lamp at dusk.",\n    "Write one short sentence about washing a cup.",\n    "Write one short sentence about reading a map.",\n    "Write one short sentence about hearing distant traffic.",\n    "Write one short sentence about folding a towel.",\n    "Write one short sentence about checking the time.",\n    "Write one short sentence about a garden path.",\n    "Write one short sentence about closing a drawer.",\n    "Write one short sentence about sharpening a pencil.",\n    "Write one short sentence about waiting for an elevator.",\n)\n\nACTIVE_WORDS = (" move", " act", " forward", " begin", " go", " start")\nCALM_WORDS = (" rest", " calm", " quiet", " wait", " pause", " still")\nPROCEED_WORDS = (" proceed", " continue", " advance", " move", " begin", " start")\nHESITATE_WORDS = (" hesitate", " delay", " pause", " wait", " stop", " hold")\n\nCONTRAST_BANKS = {\n    "action_calm": (ACTIVE_WORDS, CALM_WORDS),\n    "proceed_hesitate": (PROCEED_WORDS, HESITATE_WORDS),\n}\n\n\ndef dot(a: Sequence[float], b: Sequence[float]) -> float:\n    return sum(x * y for x, y in zip(a, b))\n\n\ndef norm(vector: Sequence[float]) -> float:\n    return math.sqrt(dot(vector, vector))\n\n\ndef normalize(vector: Sequence[float]) -> list[float]:\n    length = norm(vector)\n    if not length:\n        raise ValueError("cannot normalize zero vector")\n    return [value / length for value in vector]\n\n\ndef orthogonal_random_direction(reference: Sequence[float], seed: int) -> list[float]:\n    unit_reference = normalize(reference)\n    rng = random.Random(seed)\n    candidate = [rng.gauss(0.0, 1.0) for _ in unit_reference]\n    projection = dot(candidate, unit_reference)\n    orthogonal = [value - projection * axis for value, axis in zip(candidate, unit_reference)]\n    return normalize(orthogonal)\n\n\ndef linear_slope(xs: Sequence[float], ys: Sequence[float]) -> float:\n    x_mean = sum(xs) / len(xs)\n    y_mean = sum(ys) / len(ys)\n    denominator = sum((x - x_mean) ** 2 for x in xs)\n    if denominator == 0:\n        raise ValueError("slope requires varying x")\n    return sum((x - x_mean) * (y - y_mean) for x, y in zip(xs, ys)) / denominator\n\n\ndef exact_sign_flip_p(values: Sequence[float]) -> float:\n    observed = abs(sum(values) / len(values))\n    exceed = 0\n    total = 1 << len(values)\n    for mask in range(total):\n        permuted = sum((-value if mask & (1 << i) else value) for i, value in enumerate(values)) / len(values)\n        exceed += abs(permuted) >= observed - 1e-12\n    return exceed / total\n\n\ndef keyword_counts(text: str, positive_words: Sequence[str], negative_words: Sequence[str]) -> dict[str, int]:\n    lowered = " " + text.lower()\n    return {\n        "positive_word_count": sum(lowered.count(word) for word in positive_words),\n        "negative_word_count": sum(lowered.count(word) for word in negative_words),\n    }\n',encoding='utf-8')


In [ ]:
(WORK/'run_intervention.py').write_text('from __future__ import annotations\n\nimport argparse\nimport json\nfrom pathlib import Path\n\nfrom intervention_core import (\n    ACTIVE_WORDS,\n    CALM_WORDS,\n    CONTRAST_BANKS,\n    PROMPTS,\n    keyword_counts,\n    normalize,\n    orthogonal_random_direction,\n)\n\nMODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"\nSEMANTIC_ALPHAS = (-2.0, -1.0, -0.5, 0.0, 0.5, 1.0, 2.0)\nRANDOM_ALPHAS = (-2.0, 2.0)\n\n\ndef decoder_layers(model):\n    if hasattr(model, "model") and hasattr(model.model, "layers"):\n        return model.model.layers\n    raise ValueError("unsupported decoder layout")\n\n\nclass SteeringHook:\n    def __init__(self, torch_module, direction: list[float], alpha: float):\n        self.torch = torch_module\n        self.direction = direction\n        self.alpha = alpha\n\n    def __call__(self, module, inputs, output):\n        hidden = output[0] if isinstance(output, tuple) else output\n        direction = self.torch.tensor(self.direction, device=hidden.device, dtype=hidden.dtype)\n        edited = hidden.clone()\n        edited[:, -1, :] = edited[:, -1, :] + self.alpha * direction\n        return (edited, *output[1:]) if isinstance(output, tuple) else edited\n\n\ndef token_ids(tokenizer, words: tuple[str, ...]) -> list[int]:\n    ids = []\n    for word in words:\n        encoded = tokenizer.encode(word, add_special_tokens=False)\n        if encoded:\n            ids.append(int(encoded[0]))\n    return sorted(set(ids))\n\n\ndef parse_int_list(raw: str | None) -> list[int]:\n    if not raw:\n        return []\n    values = []\n    for part in raw.split(","):\n        part = part.strip()\n        if part:\n            values.append(int(part))\n    return values\n\n\ndef contrast_fields(bank_name: str) -> tuple[str, str, str]:\n    return (\n        f"logit_contrast_{bank_name}",\n        f"positive_word_count_{bank_name}",\n        f"negative_word_count_{bank_name}",\n    )\n\n\ndef main() -> None:\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--direction-json", type=Path, required=True)\n    parser.add_argument("--mode", choices=("smoke", "full", "layer_scan"), default="smoke")\n    parser.add_argument("--output-dir", type=Path, required=True)\n    parser.add_argument("--prompt-count", type=int, default=None)\n    parser.add_argument("--random-count", type=int, default=None)\n    parser.add_argument("--layers", type=str, default=None)\n    parser.add_argument("--contrast-bank", choices=tuple(CONTRAST_BANKS), default="proceed_hesitate")\n    args = parser.parse_args()\n\n    import torch\n    from transformers import AutoModelForCausalLM, AutoTokenizer\n\n    metadata = json.loads(args.direction_json.read_text(encoding="utf-8"))\n    semantic_direction = normalize(metadata["raw_delta"])\n    prompt_count = args.prompt_count if args.prompt_count is not None else (6 if args.mode == "smoke" else len(PROMPTS))\n    random_count = args.random_count if args.random_count is not None else (5 if args.mode == "smoke" else 63)\n    directions = [("semantic", 0, semantic_direction)]\n    directions.extend(\n        ("random", index, orthogonal_random_direction(semantic_direction, 20260703 + index))\n        for index in range(random_count)\n    )\n    layer_list = parse_int_list(args.layers)\n\n    device = "cuda" if torch.cuda.is_available() else "cpu"\n    dtype = torch.float16 if device == "cuda" else torch.float32\n    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)\n    model = AutoModelForCausalLM.from_pretrained(\n        MODEL_NAME,\n        torch_dtype=dtype,\n        device_map="auto" if device == "cuda" else None,\n    )\n    if device == "cpu":\n        model.to(device)\n    model.eval()\n    layers = decoder_layers(model)\n    default_layer = int(metadata["decoder_layer_index"])\n    target_layers = layer_list if layer_list else [default_layer]\n    bank_token_ids = {\n        bank_name: (token_ids(tokenizer, positive), token_ids(tokenizer, negative))\n        for bank_name, (positive, negative) in CONTRAST_BANKS.items()\n    }\n    primary_contrast_field = f"logit_contrast_{args.contrast_bank}"\n\n    args.output_dir.mkdir(parents=True, exist_ok=True)\n    output_path = args.output_dir / f"intervention_{args.mode}.jsonl"\n    run_id = 0\n    with output_path.open("w", encoding="utf-8") as handle:\n        for target_layer in target_layers:\n            for direction_kind, direction_index, direction in directions:\n                alphas = SEMANTIC_ALPHAS if direction_kind == "semantic" else RANDOM_ALPHAS\n                for prompt_id in range(prompt_count):\n                    text = f"Task: {PROMPTS[prompt_id]}\\nResponse:"\n                    inputs = tokenizer(text, return_tensors="pt").to(model.device)\n                    prompt_len = int(inputs["input_ids"].shape[-1])\n                    for alpha in alphas:\n                        hook = SteeringHook(torch, direction, alpha)\n                        registered = layers[target_layer].register_forward_hook(hook)\n                        try:\n                            with torch.inference_mode():\n                                result = model(**inputs, use_cache=False, return_dict=True)\n                                logits = result.logits[0, -1, :].float()\n                                bank_logit_contrasts = {}\n                                for bank_name, (positive_ids, negative_ids) in bank_token_ids.items():\n                                    positive_logit = float(logits[positive_ids].mean().cpu())\n                                    negative_logit = float(logits[negative_ids].mean().cpu())\n                                    bank_logit_contrasts[bank_name] = positive_logit - negative_logit\n                                generated_text = ""\n                                features = {}\n                                if direction_kind == "semantic":\n                                    generated = model.generate(\n                                        **inputs,\n                                        do_sample=False,\n                                        max_new_tokens=12,\n                                        use_cache=False,\n                                        pad_token_id=tokenizer.eos_token_id,\n                                    )\n                                    generated_text = tokenizer.decode(\n                                        generated[0, prompt_len:], skip_special_tokens=True\n                                    ).strip()\n                                    for bank_name, (positive_words, negative_words) in CONTRAST_BANKS.items():\n                                        counts = keyword_counts(generated_text, positive_words, negative_words)\n                                        features[f"positive_word_count_{bank_name}"] = counts["positive_word_count"]\n                                        features[f"negative_word_count_{bank_name}"] = counts["negative_word_count"]\n                                else:\n                                    for bank_name in CONTRAST_BANKS:\n                                        features[f"positive_word_count_{bank_name}"] = 0\n                                        features[f"negative_word_count_{bank_name}"] = 0\n                        finally:\n                            registered.remove()\n                        run_id += 1\n                        row = {\n                            "run_id": run_id,\n                            "model": MODEL_NAME,\n                            "mode": args.mode,\n                            "direction_kind": direction_kind,\n                            "direction_index": direction_index,\n                            "target_layer": target_layer,\n                            "alpha": alpha,\n                            "prompt_id": prompt_id,\n                            "prompt": text,\n                            **{f"logit_contrast_{bank_name}": value for bank_name, value in bank_logit_contrasts.items()},\n                            "logit_contrast": bank_logit_contrasts[args.contrast_bank],\n                            "output": generated_text,\n                            **features,\n                        }\n                        handle.write(json.dumps(row, ensure_ascii=False) + "\\n")\n                        print(\n                            f"{run_id:04d} layer={target_layer:02d} {direction_kind}:{direction_index:02d} "\n                            f"prompt={prompt_id:02d} alpha={alpha:+.1f} "\n                            f"contrast={row[primary_contrast_field]:+.3f}"\n                        )\n    print(output_path)\n\n\nif __name__ == "__main__":\n    main()\n',encoding='utf-8')


In [ ]:
(WORK/'inspect_prompt_tokens.py').write_text('from __future__ import annotations\n\nimport argparse\nimport csv\nfrom pathlib import Path\n\nfrom intervention_core import PROMPTS\nfrom run_intervention import MODEL_NAME\n\n\ndef parse_int_list(raw: str) -> list[int]:\n    return [int(part.strip()) for part in raw.split(",") if part.strip()]\n\n\ndef main() -> None:\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--positions", type=str, default="-12,-10,-8,-6,-4,-2,-1")\n    parser.add_argument("--prompt-count", type=int, default=12)\n    parser.add_argument("--output-dir", type=Path, required=True)\n    args = parser.parse_args()\n\n    from transformers import AutoTokenizer\n\n    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)\n    positions = parse_int_list(args.positions)\n    rows = []\n    for prompt_id, prompt in enumerate(PROMPTS[: args.prompt_count]):\n        text = f"Task: {prompt}\\nResponse:"\n        token_ids = tokenizer(text, add_special_tokens=False)["input_ids"]\n        tokens = tokenizer.convert_ids_to_tokens(token_ids)\n        for position in positions:\n            token_index = len(tokens) + position if position < 0 else position\n            token = tokens[token_index] if 0 <= token_index < len(tokens) else "<OUT>"\n            rows.append(\n                {\n                    "prompt_id": prompt_id,\n                    "token_count": len(tokens),\n                    "token_position": position,\n                    "token_index": token_index,\n                    "token": token,\n                    "prompt": prompt,\n                    "full_text": text,\n                }\n            )\n\n    args.output_dir.mkdir(parents=True, exist_ok=True)\n    csv_path = args.output_dir / "prompt_position_tokens.csv"\n    with csv_path.open("w", encoding="utf-8", newline="") as handle:\n        writer = csv.DictWriter(handle, fieldnames=list(rows[0]))\n        writer.writeheader()\n        writer.writerows(rows)\n\n    report = [\n        "# Qwen Prompt Position Tokens",\n        "",\n        f"- model: `{MODEL_NAME}`",\n        f"- prompt count: {args.prompt_count}",\n        f"- positions: {\', \'.join(str(position) for position in positions)}",\n        "",\n        "| prompt | token count | position | index | token |",\n        "|---:|---:|---:|---:|---|",\n    ]\n    for row in rows:\n        report.append(\n            "| {prompt_id} | {token_count} | {token_position} | {token_index} | `{token}` |".format(**row)\n        )\n    report.extend(\n        [\n            "",\n            "Negative positions are counted from the end of the prompt sequence.",\n            "The full prompt text is available in `prompt_position_tokens.csv`.",\n        ]\n    )\n    report_path = args.output_dir / "prompt_position_tokens.md"\n    report_path.write_text("\\n".join(report) + "\\n", encoding="utf-8")\n    print(report_path)\n\n\nif __name__ == "__main__":\n    main()\n',encoding='utf-8')


## Inspect Position Tokens

In [ ]:
!cd /content/neurostate_qwen_causal_intervention_v0_9 && python inspect_prompt_tokens.py --positions=-12,-10,-8,-6,-4,-2,-1 --prompt-count 12 --output-dir token_inspect


In [ ]:
from IPython.display import Markdown,display
report=WORK/'token_inspect'/'prompt_position_tokens.md'
display(Markdown(report.read_text()))


In [ ]:
import shutil
from google.colab import files
archive=shutil.make_archive('/content/neurostate_qwen_causal_intervention_v0_9_results','zip',WORK)
files.download(archive)
